# Load Environment Variables in Python and test

The following function reads your `~/.bashrc` file and sets each variable in Python’s runtime environment (`os.environ`), so you can use them directly in your code.

Any other notebook can reuse this function to reload environment variables at the start.

In [1]:
import os

with open(os.path.expanduser("~/.bashrc"), 'r') as f:
    for line in f:
        if line.strip().startswith('export '):
            parts = line.strip().replace('export ', '').split('=', 1)
            if len(parts) == 2:
                var_name = parts[0].strip()
                var_value = parts[1].strip().strip("'\"")
                
                # SKIP PATH completely!
                if var_name == 'PATH':
                    continue  # Skip this line
                    
                os.environ[var_name] = var_value

In [2]:
# Now you can access them directly:
print(f"GOOGLE_CLOUD_PROJECT = {os.environ.get('GOOGLE_CLOUD_PROJECT')}")
print(f"WORKSPACE_BUCKET = {os.environ.get('WORKSPACE_BUCKET')}")

GOOGLE_CLOUD_PROJECT = wb-cheerful-papaya-1705
WORKSPACE_BUCKET = gs://workspace-bucket-wb-cheerful-papaya-1705


In [3]:
!echo $WORKSPACE_CDR

wb-silky-artichoke-2408.C2024Q3R8


In [4]:
!echo $WORKSPACE_TEMP_BUCKET

gs://temporary-workspace-bucket-wb-cheerful-papaya-1705


In [6]:
import os

# Define the variables to capture from your Notebook VM
vars_to_capture = [
    "WORKSPACE_CDR", 
    "GOOGLE_CLOUD_PROJECT", 
    "WORKSPACE_BUCKET", 
    "WORKSPACE_TEMP_BUCKET",
    "EXPORT_BUCKET" # Adding the 5th variable
]

r_file_path = "/home/jupyter/repos/aou_disease_prediction_2/aou_scripts/Setting_Env_Variables_p2.R"

with open(r_file_path, "w") as f:
    f.write("# --- AUTO-GENERATED ENVIRONMENT SETUP ---\n\n")
    
    # Part 1: Write the Sys.setenv lines
    for var in vars_to_capture:
        # Get value from environment
        value = os.environ.get(var)
        
        # Logic: If EXPORT_BUCKET is empty, use WORKSPACE_BUCKET value
        if var == "EXPORT_BUCKET" and not value:
            value = os.environ.get("WORKSPACE_BUCKET", "NOT_FOUND")
            
        # Fallback if the variable doesn't exist at all
        if not value:
            value = "NOT_FOUND"
            
        f.write(f'Sys.setenv({var} = "{value}")\n')
    
    # Part 2: Verification Block
    f.write("\n# --- VERIFICATION BLOCK ---\n")
    
    # Format the list for R: "VAR1", "VAR2", etc.
    r_list_str = ", ".join([f'"{v}"' for v in vars_to_capture])
    
    verification_code = f"""
vars_to_check <- c({r_list_str})

cat("\\n🔍 Current Workspace Variables:\\n")
for (v in vars_to_check) {{
  value <- Sys.getenv(v)
  # Prints the name (padded to 22 chars) and the value
  cat(sprintf("  %-22s : %s\\n", v, value))
}}
message("\\n✅ Environment Loaded Successfully.")
"""
    f.write(verification_code)

print(f"Done! {r_file_path} now contains {len(vars_to_capture)} variables.")

Done! /home/jupyter/workspace/aou-tutorial-notebooks/Setting_Env_Variables_p2.R now contains 5 variables.
